<a href="https://colab.research.google.com/github/lestermartin/Security_Labs/blob/master/workshops/ibis-pipelines/IbisWithStarburstGalaxy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Building Data Pipelines with Ibis and Starburst

This notebook represents the hands-on portion of the [Building Data Pipelines with Ibis and Starburst](https://github.com/starburstdata/devrel/tree/main/workshops/ibis-pipelines) webinar.

**Follow the [INSTRUCTIONS](https://github.com/starburstdata/devrel/blob/main/workshops/ibis-pipelines/INSTRUCTIONS.md) to get setup with Starburst Galaxy and a writeable catalog leveraging an S3 bucket.**


## Set YOUR env properties

The examples in the remainder of this notebook where build while using [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) (a super fast way to get going with Trino) using it's simple username/password authentication.  

**As stated above, follow the [INSTRUCTIONS](https://github.com/starburstdata/devrel/blob/main/workshops/ibis-pipelines/INSTRUCTIONS.md) to get setup with Starburst Galaxy and a writeable catalog leveraging an S3 bucket.**

To obtain the host name and your full user name, log into YOUR [Starburst Galaxy tenant](https://galaxy.starburst.io/login) and follow these steps.

1. From the *left-hand nav*, click on **Admin** > **Clusters**
2. From the **Clusters** list, scroll to the far right of the cluster you will be using and click on the *vertical elipsis* icon
3. Select **Partner connect** from the *contextual menu* that surfaces
4. Scroll down to the **Drivers & Clients** section and click on the Ibis tile
5. Use the **User** and **Host** properties present (plus your password) in the next cell

In [3]:
import getpass

# grab credentials from the notebook user to be used when making a connection
my_host = input("Host name")
my_username = input("User name")
my_password = getpass.getpass("Password")

Host namelester-aws-us-east-1-free.trino.galaxy.starburst.io
User namelester.martin@starburstdata.com/accountadmin
Password··········


## Exploring Ibis

Python DataFrame API as described at the [Ibis project website](https://ibis-project.org/).

NOTE: Ibis can run against many different SQL engines, not just Trino.

### Env setup

In [1]:
# install Ibis

%pip install trino
%pip install 'ibis-framework[trino]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 15.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [4]:
# boiler-plate code for setup

import os
import ibis
from trino.auth import BasicAuthentication

ibis.options.interactive = True

user = my_username
trino_auth_obj = BasicAuthentication(my_username, my_password)
host = my_host
port = "443"
http_scheme = "https"
catalog = "tmp_cat"
schema = "tmp_lester_tester_12345"  #"tmp_first_last_postalcode" # don't forget to CHANGE this!

con = ibis.trino.connect(
    user=user, auth=trino_auth_obj, host=host, port=port, http_scheme=http_scheme, database=catalog, schema=schema
)

print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')
con.sql("select '** CONNECTION IS GOOD **' as conn_check")


 Make sure the phrase ** CONNECTION IS GOOD ** displays 



┏━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ conn_check               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string                   │
├──────────────────────────┤
│ ** CONNECTION IS GOOD ** │
└──────────────────────────┘

### Interactive exploration

You could slide into your problem solving by take baby steps to see what each bit is doing for you.

Note: This code was originally published in  [ibis & trino (dataframe api part deux)](https://lestermartin.blog/2023/10/27/ibis-trino-dataframe-api-part-deux/).  

In [16]:
# select a full table

custDF = con.table("customer", database=("tpch", "tiny"))
custDF[0:10]

┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ custkey ┃ name               ┃ address                              ┃ nationkey ┃ phone           ┃ acctbal  ┃ mktsegment ┃ comment                                                                          ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ !int64  │ !string            │ !string                              │ !int64    │ !string         │ !float64 │ !string    │ !string                                                                          │
├─────────┼────────────────────┼──────────────────────────────────────┼───────────┼─────────────────┼──────────┼────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│     751 │ Customer#000000751 │ e OSrreG6sx7l1t3wAg8u11DWk D 9       │         0 │ 10-658-550-2257 │  2130.98 │ FURNITURE  │ ges sleep furiously bold deposits. furiously regular requests cajole slyly. unu… │
│     752 │ Customer#000000752 │ KtdEacPUecPdPLt99kwZrnH9oIxUxpw      │         8 │ 18-924-993-6038 │  8363.66 │ MACHINERY  │ mong the ironic, final waters. regular deposits above the fluffily ironic instr… │
│     753 │ Customer#000000753 │ 9k2PLlDRbMq4oSvW5Hh7Ak5iRDH          │        17 │ 27-817-126-3646 │  8114.44 │ HOUSEHOLD  │ cies. deposits snooze. final, regular excuses wake furiously about the furiousl… │
│     754 │ Customer#000000754 │ 8r5wwhhlL9MkAxOhRK                   │         0 │ 10-646-595-5871 │  -566.86 │ BUILDING   │ er regular accounts against the furiously unusual somas sleep carefull           │
│     755 │ Customer#000000755 │ F2YYbRT2EV                           │        16 │ 26-395-247-2207 │  7631.94 │ HOUSEHOLD  │ xpress instructions breach; pending request                                      │
│     756 │ Customer#000000756 │ Lv7cG by4Wyd8Hzmumwp8hSIZg9          │        14 │ 24-267-298-7503 │  8116.99 │ AUTOMOBILE │ ly unusual deposits. fluffily express deposits nag blithely above the silent, e… │
│     757 │ Customer#000000757 │ VFnouow3LhLvEDy                      │         3 │ 13-704-408-2991 │  9334.82 │ AUTOMOBILE │ riously furiously unusual asymptotes. slyly                                      │
│     758 │ Customer#000000758 │ 8fJLXfS5Zup0GQ3xBKL3eAC Q            │        17 │ 27-175-799-9168 │  6352.14 │ HOUSEHOLD  │ eposits. blithely unusual deposits affix care                                    │
│     759 │ Customer#000000759 │ IX1uj4NFhOmu0V xDtiYzHVzWfi8bl,5EHtJ │         1 │ 11-731-806-1019 │  3477.59 │ FURNITURE  │ above the quickly pending requests nag final, ex                                 │
│     760 │ Customer#000000760 │ jp8DYJ7GPQSDQC                       │         2 │ 12-176-116-3113 │  2883.24 │ BUILDING   │ uriously alongside of the ironic deposits. slyly thin pinto beans a              │
└─────────┴────────────────────┴──────────────────────────────────────┴───────────┴─────────────────┴──────────┴────────────┴──────────────────────────────────────────────────────────────────────────────────┘

In [15]:
# use projection

projectedDF = custDF.select("name", "acctbal", "nationkey")
projectedDF[0:10]

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ name               ┃ acctbal  ┃ nationkey ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ !string            │ !float64 │ !int64    │
├────────────────────┼──────────┼───────────┤
│ Customer#000000001 │   711.56 │        15 │
│ Customer#000000002 │   121.65 │        13 │
│ Customer#000000003 │  7498.12 │         1 │
│ Customer#000000004 │  2866.83 │         4 │
│ Customer#000000005 │   794.47 │         3 │
│ Customer#000000006 │  7638.57 │        20 │
│ Customer#000000007 │  9561.95 │        18 │
│ Customer#000000008 │  6819.74 │        17 │
│ Customer#000000009 │  8324.07 │         8 │
│ Customer#000000010 │  2753.54 │         5 │
│ …                  │        … │         … │
└────────────────────┴──────────┴───────────┘

In [25]:
# apply a predicate

filteredDF = projectedDF.filter(projectedDF["acctbal"] > 50.0)
filteredDF[0:100]

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ name               ┃ acctbal  ┃ nationkey ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ !string            │ !float64 │ !int64    │
├────────────────────┼──────────┼───────────┤
│ Customer#000000001 │   711.56 │        15 │
│ Customer#000000002 │   121.65 │        13 │
│ Customer#000000003 │  7498.12 │         1 │
│ Customer#000000004 │  2866.83 │         4 │
│ Customer#000000005 │   794.47 │         3 │
│ Customer#000000006 │  7638.57 │        20 │
│ Customer#000000007 │  9561.95 │        18 │
│ Customer#000000008 │  6819.74 │        17 │
│ Customer#000000009 │  8324.07 │         8 │
│ Customer#000000010 │  2753.54 │         5 │
│ …                  │        … │         … │
└────────────────────┴──────────┴───────────┘

In [26]:
# select a second table

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )
nationDF[0:10]

┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ n_nationkey ┃ nation_name ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ !int64      │ !string     │
├─────────────┼─────────────┤
│           0 │ ALGERIA     │
│           1 │ ARGENTINA   │
│           2 │ BRAZIL      │
│           3 │ CANADA      │
│           4 │ EGYPT       │
│           5 │ ETHIOPIA    │
│           6 │ FRANCE      │
│           7 │ GERMANY     │
│           8 │ INDIA       │
│           9 │ INDONESIA   │
└─────────────┴─────────────┘

In [32]:
# join the tables

joinedDF = filteredDF.join(nationDF,
    filteredDF.nationkey == nationDF.n_nationkey)
joinedDF[0:10]

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ name               ┃ acctbal ┃ nationkey ┃ n_nationkey ┃ nation_name ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string             │ float64 │ int64     │ int64       │ string      │
├────────────────────┼─────────┼───────────┼─────────────┼─────────────┤
│ Customer#000000751 │ 2130.98 │         0 │           0 │ ALGERIA     │
│ Customer#000000752 │ 8363.66 │         8 │           8 │ INDIA       │
│ Customer#000000753 │ 8114.44 │        17 │          17 │ PERU        │
│ Customer#000000755 │ 7631.94 │        16 │          16 │ MOZAMBIQUE  │
│ Customer#000000756 │ 8116.99 │        14 │          14 │ KENYA       │
│ Customer#000000757 │ 9334.82 │         3 │           3 │ CANADA      │
│ Customer#000000758 │ 6352.14 │        17 │          17 │ PERU        │
│ Customer#000000759 │ 3477.59 │         1 │           1 │ ARGENTINA   │
│ Customer#000000760 │ 2883.24 │         2 │           2 │ BRAZIL      │
│ Customer#000000761 │ 1525.96 │        19 │          19 │ ROMANIA     │
└────────────────────┴─────────┴───────────┴─────────────┴─────────────┘

In [33]:
# project the joined results

projectedJoinDF = joinedDF.drop("nationkey", "n_nationkey")
projectedJoinDF[0:10]

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ name               ┃ acctbal ┃ nation_name ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string             │ float64 │ string      │
├────────────────────┼─────────┼─────────────┤
│ Customer#000000751 │ 2130.98 │ ALGERIA     │
│ Customer#000000752 │ 8363.66 │ INDIA       │
│ Customer#000000753 │ 8114.44 │ PERU        │
│ Customer#000000755 │ 7631.94 │ MOZAMBIQUE  │
│ Customer#000000756 │ 8116.99 │ KENYA       │
│ Customer#000000757 │ 9334.82 │ CANADA      │
│ Customer#000000758 │ 6352.14 │ PERU        │
│ Customer#000000759 │ 3477.59 │ ARGENTINA   │
│ Customer#000000760 │ 2883.24 │ BRAZIL      │
│ Customer#000000761 │ 1525.96 │ ROMANIA     │
└────────────────────┴─────────┴─────────────┘

In [36]:
# do some aggregations

groupedDF = projectedJoinDF.group_by("nation_name").aggregate(
    nbr_custs   = projectedJoinDF.count(),
    avg_acctbal = projectedJoinDF.acctbal.mean()
)
groupedDF[0:10]



┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ nation_name   ┃ nbr_custs ┃ avg_acctbal ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string        │ int64     │ float64     │
├───────────────┼───────────┼─────────────┤
│ PERU          │        51 │ 4776.530980 │
│ MOZAMBIQUE    │        56 │ 5143.720000 │
│ CANADA        │        58 │ 5002.291379 │
│ ROMANIA       │        55 │ 4638.587455 │
│ SAUDI ARABIA  │        63 │ 5886.196508 │
│ CHINA         │        55 │ 5357.039636 │
│ ETHIOPIA      │        50 │ 4106.757000 │
│ IRAN          │        66 │ 4633.657121 │
│ GERMANY       │        53 │ 4646.423774 │
│ UNITED STATES │        42 │ 4968.700238 │
└───────────────┴───────────┴─────────────┘

In [38]:
# add sorting

orderedDF = groupedDF.order_by([ibis.desc("avg_acctbal")])
orderedDF[0:10]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ nation_name  ┃ nbr_custs ┃ avg_acctbal ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string       │ int64     │ float64     │
├──────────────┼───────────┼─────────────┤
│ SAUDI ARABIA │        63 │ 5886.196508 │
│ MOROCCO      │        68 │ 5837.108971 │
│ INDONESIA    │        60 │ 5511.938000 │
│ KENYA        │        45 │ 5479.570000 │
│ CHINA        │        55 │ 5357.039636 │
│ JAPAN        │        63 │ 5308.519524 │
│ IRAQ         │        51 │ 5306.893529 │
│ ARGENTINA    │        55 │ 5243.712727 │
│ MOZAMBIQUE   │        56 │ 5143.720000 │
│ VIETNAM      │        54 │ 5094.512407 │
└──────────────┴───────────┴─────────────┘

#### Leverage lazy execution

The baby steps above were great for figuring out your logic, but if you focus only getting the final answer, all of the DataFrame transformations will simply wait until you have a final I/O action triggered.

In [40]:
# put it all together

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )

custDF = con.table("customer", database=("tpch", "tiny")) \
            .select("name", "acctbal", "nationkey") \
            .filter(projectedDF["acctbal"] > 50.0)

apiSQL = custDF.join(nationDF,
    custDF.nationkey == nationDF.n_nationkey) \
            .drop("nationkey", "n_nationkey") \
            .group_by("nation_name") \
            .aggregate(
                nbr_custs   = projectedJoinDF.count(),
                avg_acctbal = projectedJoinDF.acctbal.mean()
            ) \
            .order_by([ibis.desc("avg_acctbal")])

apiSQL[0:10]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ nation_name  ┃ nbr_custs ┃ avg_acctbal ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string       │ int64     │ float64     │
├──────────────┼───────────┼─────────────┤
│ SAUDI ARABIA │        63 │ 5886.196508 │
│ MOROCCO      │        68 │ 5837.108971 │
│ INDONESIA    │        60 │ 5511.938000 │
│ KENYA        │        45 │ 5479.570000 │
│ CHINA        │        55 │ 5357.039636 │
│ JAPAN        │        63 │ 5308.519524 │
│ IRAQ         │        51 │ 5306.893529 │
│ ARGENTINA    │        55 │ 5243.712727 │
│ MOZAMBIQUE   │        56 │ 5143.720000 │
│ VIETNAM      │        54 │ 5094.512407 │
└──────────────┴───────────┴─────────────┘

#### SQL is available, too

Ibis strives to make their DataFrame API be the only (or at least primary) way you run your data operations, it sometimes it just might be better to write the SQL to get a different result and/or process to run.

NOTE: There could be a performance difference as Ibis is not doing anything with the SQL you write -- it just submits it like it is. Additionally, this approach hinders the optionality promise of Ibis due to SQL nuances among different SQL engines.

In [45]:
# or... just run some SQL

sqlDF = con.sql("""
         SELECT n.name AS nation_name,
                COUNT() AS nbr_custs,
                AVG(c.acctbal) AS avg_acctbal
           FROM tpch.tiny.customer c
           JOIN tpch.tiny.nation n
             ON c.nationkey = n.nationkey
          WHERE c.acctbal > 50.0
          GROUP BY n.name
          ORDER BY AVG(c.acctbal) DESC
""")
sqlDF[0:10]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ nation_name  ┃ nbr_custs ┃ avg_acctbal ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string       │ int64     │ float64     │
├──────────────┼───────────┼─────────────┤
│ SAUDI ARABIA │        63 │ 5886.196508 │
│ MOROCCO      │        68 │ 5837.108971 │
│ INDONESIA    │        60 │ 5511.938000 │
│ KENYA        │        45 │ 5479.570000 │
│ CHINA        │        55 │ 5357.039636 │
│ JAPAN        │        63 │ 5308.519524 │
│ IRAQ         │        51 │ 5306.893529 │
│ ARGENTINA    │        55 │ 5243.712727 │
│ MOZAMBIQUE   │        56 │ 5143.720000 │
│ VIETNAM      │        54 │ 5094.512407 │
└──────────────┴───────────┴─────────────┘

## Real-world example

Let's use the Burst Bank customer table as if it is our raw source already ingested. We can focus on doing some limited quality checks, and transformations, as well as a precalculated aggregates table.

### Explore the data

Notice how the optional `database=("<catalog>", "<schema>")` parameter allows you to select something different than was configured in the `con` object.

Seems a good idea to always being this specific; especially for code to be shared and/or productionalized.

Let's start with the `customer` table.

In [46]:

bank_customer = con.table("customer", database=("sample", "burstbank"))
bank_customer[0:20]

┏━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ custkey ┃ first_name ┃ last_name ┃ street                         ┃ city             ┃ state  ┃ postcode ┃ country ┃ phone                 ┃ dob        ┃ gender ┃ married ┃ ssn         ┃ paycheck_dd ┃ estimated_income ┃ fico  ┃ registration_date ┃
┡━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ string  │ string     │ string    │ string                         │ string           │ string │ string   │ string  │ string                │ string     │ string │ string  │ string      │ string      │ float64          │ int32 │ string            │
├─────────┼────────────┼───────────┼────────────────────────────────┼──────────────────┼────────┼──────────┼─────────┼───────────────────────┼────────────┼────────┼─────────┼─────────────┼─────────────┼──────────────────┼───────┼───────────────────┤
│ 1000001 │ Jessica    │ Wilkins   │ 6482 Lopez Run                 │ Huynhburgh       │ NC     │ 39644    │ US      │ +1-241-619-2638x950   │ 1952-03-23 │ M      │ N       │ 123-64-3546 │ N           │         51299.55 │   490 │ 1966-09-08        │
│ 1000002 │ Ryan       │ Moon      │ 85265 Bruce Lock               │ East Tonyafort   │ DC     │ 17523    │ US      │ (224)281-8020         │ 1988-02-14 │ M      │ N       │ 190-78-1571 │ N           │        168586.71 │   570 │ 2000-05-11        │
│ 1000003 │ Richard    │ Tyler     │ 62298 Choi Mission             │ East Natalie     │ VT     │ 60907    │ US      │ 605.889.1125          │ 1925-06-01 │ M      │ N       │ 282-96-7276 │ Y           │        253388.29 │   677 │ 1986-04-17        │
│ 1000004 │ Shane      │ Baker     │ 3672 Jason Greens              │ Connieside       │ YT     │ L6G3A1   │ CA      │ 349-852-4011          │ 1992-05-31 │ F      │ Y       │ 213-71-8261 │ N           │        372036.69 │   719 │ 2002-12-03        │
│ 1000005 │ Kimberly   │ Hernandez │ 9057 Luis Junctions Apt. 699   │ North Paul       │ NT     │ N7T8C1   │ CA      │ 001-389-832-7031x1811 │ 1956-09-15 │ U      │ U       │ 581-31-3500 │ N           │        135704.21 │   584 │ 1962-04-10        │
│ 1000006 │ Mitchell   │ Barker    │ 8088 Richardson Keys Suite 671 │ East Shannon     │ OK     │ 87423    │ US      │ +1-753-406-5384x3134  │ 1996-10-13 │ M      │ N       │ 006-03-2480 │ N           │        100559.42 │   436 │ 1976-04-20        │
│ 1000007 │ David      │ Lynch     │ 726 Corey Turnpike Apt. 566    │ Shawmouth        │ ON     │ H7E 8K9  │ CA      │ 729.594.8277x07536    │ 1984-10-26 │ F      │ Y       │ 231-79-8327 │ N           │        250508.69 │   648 │ 1998-07-06        │
│ 1000008 │ Margaret   │ Larson    │ 8949 Collins Harbor            │ North Donnatown  │ NB     │ H1B2E7   │ CA      │ 2669602288            │ 1940-03-30 │ F      │ Y       │ 504-14-5840 │ N           │         61472.00 │   376 │ 1952-10-31        │
│ 1000009 │ Charles    │ Jefferson │ 07242 Moore Rue Apt. 440       │ East Thomasmouth │ NT     │ J9Y 6H7  │ CA      │ (321)624-3156x89435   │ 1973-10-03 │ F      │ Y       │ 318-01-4779 │ N           │        351944.26 │   743 │ 1992-12-22        │
│ 1000010 │ Todd       │ Garcia    │ 983 Gregory Avenue             │ North Gregville  │ NC     │ 62701    │ US      │ 116.275.7286          │ 1954-06-07 │ M      │ N       │ 421-19-7088 │ N           │        316279.09 │   665 │ 1962-04-06        │
│ …       │ …          │ …         │ …                              │ …                │ …      │ …        │ …       │ …                     │ …          │ …      │ …       │ …           │ …           │                … │     … │ …                 │


#### Addresses

A quick glance suggests there are US and CA addresses which already (further) suggests that `state` is actually US-oriented.

Let's see if the country/state combos seem to make sense.

In [47]:
state_groups = bank_customer.group_by(["country", "state"]).aggregate(
    nbr_custs = bank_customer.custkey.count(),
    avg_fico  = bank_customer.fico.mean()
).order_by("country", "state")

state_groups.to_pandas().head(2000)

,country,state,nbr_custs,avg_fico
0,CA,AB,31,605.225806
1,CA,BC,47,560.936170
2,CA,MB,36,549.083333
3,CA,NB,36,584.583333
4,CA,NL,49,565.816327
...,...,...,...,...
62,US,VT,11,558.909091
63,US,WA,12,579.666667
64,US,WI,10,628.400000
65,US,WV,12,648.333333


67 rows sounds a little high as there are 50 states in the US and something less than 17 for Canadian provinces. Upon deeper look, this includes Canadian territories and US military designators for those serving overseas.

Generally, the data is PROBABLY good so we can leave it alone. Probably would want to verify that the upstream system is validating & standardizing addresses at capture time, but always a good data quality check. Could even build some error handling if it seemed necessary.

#### Phone numbers

The `phone` data looks to be all over the place. Let's install a phone number checking library and do a simple hard-coded test to make sure it works.

In [48]:
%pip install phonenumbers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.5 MB/s eta 0:00:00


In [59]:
import phonenumbers

def standardize_phone(raw: str, region: str = "US") -> str:
    parsed = phonenumbers.parse(raw, region)

    if not phonenumbers.is_valid_number(parsed):
        raise ValueError(f"Invalid phone number: {raw}")

    return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)


# Examples
numbers = [
    "2025551234",
    "(202) 555-1234",
    "(202) 555-1234 ext. 103", # notice how extensions get lost
    "202-555-1234x103",
    "202.555.1234",
    "+1 202 555 1234",
    "1-202-555-1234",
]

for n in numbers:
    print(f"{n:25} -> {standardize_phone(n)}")

2025551234                -> +12025551234
(202) 555-1234            -> +12025551234
(202) 555-1234 ext. 103   -> +12025551234
202-555-1234x103          -> +12025551234
202.555.1234              -> +12025551234
+1 202 555 1234           -> +12025551234
1-202-555-1234            -> +12025551234


Pull out the phone number column from the dataframe and run the values against the generator to see what kind of shape this data is in.

In [60]:
import pandas as pd
import phonenumbers

# quick wrapper around the phonenumbers library
def standardize_phone(raw: str, region: str = "US") -> str:
    try:
        parsed = phonenumbers.parse(str(raw), region)
        if not phonenumbers.is_valid_number(parsed):
            return None  # or return raw to keep original
        return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)
    except phonenumbers.NumberParseException:
        return None  # unparseable values return None

# Trino does NOT support this kind of basic Python UDF (needs SQL)
phone_pdf = bank_customer.select("phone").to_pandas()

# With the pandas DF create a new column for the cleaned up values
phone_pdf["phone_std"] = phone_pdf["phone"].apply(standardize_phone)
phone_pdf.head(1500)

,phone,phone_std
0,+1-241-619-2638x950,None
1,(224)281-8020,+12242818020
2,605.889.1125,+16058891125
3,349-852-4011,None
4,001-389-832-7031x1811,None
...,...,...
995,6838635316,+16838635316
996,855.199.1592,None
997,001-324-398-4995x1691,None
998,+1-524-376-1551x49724,+15243761551


Most values are not getting standardizing, but it isn't as bad as it seems as this is a generated list of numbers and most actually are invalid for one reason or another.

We did find out from the earlier test of `phonenumbers` that the extensions get tossed away.

We determined:

1. We cannot reliably perform standardization with the bad data we have at this point.
1. We need to go upstream to enhance the data collection application to be more rigorous than just capturing a string.

IF the data was in good enough shape to standardize a high-percentage of the phone numbers, we could include that as an additional column on the silver table that could be created from this bronze dataset.



#### Dates

Below we see the dates all seem to be in good shape so we'll update the cast them into our silver table.

In [58]:
check_dates = bank_customer.select("custkey", "dob", "registration_date").mutate(
    #key_casted = bank_customer.custkey.cast("date"), # uncomment to show all would blow up if bad
    dob_casted = bank_customer.dob.cast("date"),
    reg_casted = bank_customer.registration_date.cast("date")
)

print(check_dates)

┏━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ custkey ┃ dob        ┃ registration_date ┃ dob_casted ┃ reg_casted ┃
┡━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ string  │ string     │ string            │ date       │ date       │
├─────────┼────────────┼───────────────────┼────────────┼────────────┤
│ 1000001 │ 1952-03-23 │ 1966-09-08        │ 1952-03-23 │ 1966-09-08 │
│ 1000002 │ 1988-02-14 │ 2000-05-11        │ 1988-02-14 │ 2000-05-11 │
│ 1000003 │ 1925-06-01 │ 1986-04-17        │ 1925-06-01 │ 1986-04-17 │
│ 1000004 │ 1992-05-31 │ 2002-12-03        │ 1992-05-31 │ 2002-12-03 │
│ 1000005 │ 1956-09-15 │ 1962-04-10        │ 1956-09-15 │ 1962-04-10 │
│ 1000006 │ 1996-10-13 │ 1976-04-20        │ 1996-10-13 │ 1976-04-20 │
│ 1000007 │ 1984-10-26 │ 1998-07-06        │ 1984-10-26 │ 1998-07-06 │
│ 1000008 │ 1940-03-30 │ 1952-10-31        │ 1940-03-30 │ 1952-10-31 │
│ 1000009 │ 1973-10-03 │ 1992-12-22        │ 1973-10-03 │ 1992-12-22 │
│ 1000

In [ ]:
con.create_table("customers_clean2", phone_pdf, overwrite=True, database=("mycloud", "book_testing"))
print("Table created!")